# Lane 1: Symplectic Muon - Google Colab Runner
**Project:** Symplectic Muon (Hamiltonian Matrix-Orthogonalized Optimization)  
**Author:** Jaime Jarauta  
**Pre-Registered Question:** Does thermostatted symplectic kinetic scaling converge stably on an ill-conditioned spectrum and eliminate late-stage limit-cycle oscillations?

This notebook is configured for one-click execution on Google Colab (Free T4 or Pro A100/L4) or local GPUs.


In [ ]:
# Step 1: Pre-flight Hardware Verification & Credit Guard
import sys, os, time, math
import torch
from IPython import get_ipython

# Safety Fallback: Automatically unassign Colab VM on ANY error to prevent idle burn
def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Unassigning Colab GPU immediately to prevent idle burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

print(f"PyTorch Version: {torch.__version__}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    dev_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_ok = torch.cuda.is_bf16_supported()
    print(f"Device: {dev_name} | VRAM: {vram_gb:.2f} GB | bfloat16: {bf16_ok}")
    # Optimize execution for NVIDIA Tensor Cores
    torch.set_float32_matmul_precision("high")
    torch.backends.cudnn.benchmark = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Device: Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("Device: CPU")


In [ ]:
# Step 1b: Clone Patched Repository & Install Dependencies
import os, sys, subprocess
if 'google.colab' in sys.modules:
    repo_dir = 'symplectic-muon'
    if not os.path.exists(repo_dir):
        print('Cloning symplectic-muon (branch: feature/initial-implementation)...')
        subprocess.run(['git', 'clone', '-b', 'feature/initial-implementation', 'https://github.com/jaimejarauta/symplectic-muon.git'], check=True)
    if os.path.exists(repo_dir):
        os.chdir(repo_dir)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', 'wandb', 'pytest'], check=True)
    if 'src' not in sys.path:
        sys.path.insert(0, 'src')
    print('Repository ready:', os.getcwd())


In [ ]:
# Step 1c: Secure Weights & Biases Authentication
# Securely reads from Colab Secrets (Key icon on left sidebar) or prompts interactively.
# API keys are NEVER hardcoded or committed into public git repositories.
import os
try:
    from google.colab import userdata
    key = userdata.get('WANDB_API_KEY')
    if key:
        os.environ['WANDB_API_KEY'] = key
        print('[W&B] Loaded WANDB_API_KEY from Google Colab Secrets.')
except Exception:
    pass

import wandb
if 'WANDB_API_KEY' not in os.environ:
    print('[W&B] Authenticate interactively below (or add WANDB_API_KEY to Colab Secrets):')
    wandb.login()
else:
    wandb.login(key=os.environ['WANDB_API_KEY'])

os.environ['WANDB_ENTITY'] = 'research-projekts'
print('[W&B] Authenticated successfully. Target entity: research-projekts')


In [ ]:
# Step 2: Environment Setup & Fast Dependency Installation
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Configuring Google Colab environment...")
    !pip install -q uv
    !uv pip install --system -q pyyaml wandb pytest
    
    # If working directly in Colab without local git clone:
    if not os.path.exists("src"):
        # Create minimal package structure if running standalone
        os.makedirs("src/symplectic_muon", exist_ok=True)
        os.makedirs("configs", exist_ok=True)
        os.makedirs("results", exist_ok=True)

if "src" not in sys.path:
    sys.path.insert(0, "src")
print("Environment ready.")


In [ ]:
# Step 3: Fast Symplectic Muon Engine (Dual-mode: Package import with inline fallback)
try:
    from symplectic_muon.optimizer import SymplecticMuon, zeropower_via_newtonschulz5
    print("Loaded Symplectic Muon from installed package.")
except ImportError:
    print("Package not found on path; loading self-contained standalone implementation...")
    import math
    from torch.optim.optimizer import Optimizer
    
    def zeropower_via_newtonschulz5(G: torch.Tensor, steps: int = 5, eps: float = 1e-7) -> torch.Tensor:
        assert G.ndim == 2
        a, b, c = 3.4445, -4.7750, 2.0315
        if G.is_cuda and torch.cuda.is_bf16_supported():
            X = G.bfloat16()
        elif G.is_cuda:
            X = G.half()
        else:
            X = G.float()
        X = X / (X.norm() + eps)
        transposed = X.size(0) > X.size(1)
        if transposed:
            X = X.T
        for _ in range(steps):
            A = torch.mm(X, X.T)
            B = torch.addmm(A, A, A, beta=b, alpha=c)
            X = torch.addmm(X, B, X, beta=a, alpha=1.0)
        if transposed:
            X = X.T
        return X.to(G.dtype)
    
    class SymplecticMuon(Optimizer):
        def __init__(self, params, lr: float = 0.02, momentum: float = 0.95, weight_decay: float = 0.01,
                     steps: int = 5, eps: float = 1e-7, gamma_min: float = 0.5, gamma_max: float = 1.8,
                     ema_alpha: float = 0.05):
            defaults = dict(lr=lr, momentum=momentum, weight_decay=weight_decay,
                            steps=steps, eps=eps, gamma_min=gamma_min,
                            gamma_max=gamma_max, ema_alpha=ema_alpha)
            super().__init__(params, defaults)
        
        @torch.no_grad()
        def step(self, closure=None):
            loss = None
            if closure is not None:
                with torch.enable_grad(): loss = closure()
            for group in self.param_groups:
                lr, mu, wd = group['lr'], group['momentum'], group['weight_decay']
                steps, eps = group['steps'], group['eps']
                g_min, g_max, alpha = group['gamma_min'], group['gamma_max'], group['ema_alpha']
                for p in group['params']:
                    if p.grad is None: continue
                    g = p.grad
                    state = self.state[p]
                    if 'momentum_buffer' not in state:
                        state['momentum_buffer'] = torch.zeros_like(p)
                        state['running_norm'] = g.norm().item() + eps
                    buf = state['momentum_buffer']
                    buf.mul_(mu).add_(g)
                    if p.ndim == 2 and min(p.shape) > 1:
                        p_orth = zeropower_via_newtonschulz5(buf, steps=steps, eps=eps)
                        curr_norm = buf.norm().item()
                        state['running_norm'] = (1.0 - alpha) * state['running_norm'] + alpha * curr_norm
                        rel_scale = curr_norm / (state['running_norm'] + eps)
                        gamma = max(g_min, min(g_max, rel_scale))
                        d0, d1 = p.size(0), p.size(1)
                        aspect_scale = math.sqrt(max(d0, d1) / min(d0, d1))
                        update = (gamma * aspect_scale) * p_orth
                    else:
                        update = buf
                    if wd != 0: p.mul_(1.0 - lr * wd)
                    p.add_(update, alpha=-lr)
            return loss
    print("Standalone Symplectic Muon initialized.")


In [ ]:
# Step 4: Execute Smoke Test on Ill-Conditioned Spectrum
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(2026)
d_in, d_hidden, d_out = 128, 256, 64
batch_size, n_steps = 64, 500

# Synthesize ill-conditioned spectrum with geometric singular decay
X = torch.randn(batch_size * 10, d_in, device=device)
decay = torch.pow(0.8, torch.arange(d_in, device=device).float())
X = X * decay
W_true = torch.randn(d_in, d_out, device=device)
Y = X @ W_true + 0.05 * torch.randn(X.size(0), d_out, device=device)

model = nn.Sequential(
    nn.Linear(d_in, d_hidden, bias=False),
    nn.GELU(),
    nn.Linear(d_hidden, d_out, bias=False)
).to(device)

opt = SymplecticMuon(
    model.parameters(),
    lr=0.01,
    momentum=0.95,
    gamma_min=0.5,
    gamma_max=1.8,
    ema_alpha=0.05,
    steps=5
)

t0 = time.time()
losses = []
for step in range(n_steps):
    idx = (step * batch_size) % (X.size(0) - batch_size)
    x_b, y_b = X[idx:idx+batch_size], Y[idx:idx+batch_size]
    
    pred = model(x_b)
    loss = F.mse_loss(pred, y_b)
    
    opt.zero_grad()
    loss.backward()
    opt.step()
    
    losses.append(loss.item())
    if step % 100 == 0 or step == n_steps - 1:
        print(f"Step {step:03d} | Loss: {loss.item():.4f}")

elapsed = time.time() - t0
print(f"\nOptimization finished in {elapsed:.2f}s ({elapsed/n_steps*1000:.2f} ms/step).")


In [ ]:
# Step 5: Pre-Registered Gate Assessment
initial_loss = losses[0]
final_loss = losses[-1]
passed = final_loss < initial_loss * 0.10

print("=" * 60)
print(f"Initial Loss: {initial_loss:.4f} -> Final Loss: {final_loss:.4f}")
print(f"Loss Reduction: {(1.0 - final_loss/initial_loss)*100:.1f}%")
print(f"Pre-Registered Gate (final_loss < initial_loss * 0.10): {'PASSED (Keep)' if passed else 'FAILED (Kill)'}")
print("=" * 60)


In [ ]:
# Step 5b: Stage 2 NanoGPT-124M Comparative Benchmark (Muon vs Symplectic Muon)
!python scripts/train_nanogpt.py --optimizer standard_muon --steps 2000 --batch-size 16 --wandb --wandb-project research-projekts --wandb-name muon-baseline-t4
!python scripts/train_nanogpt.py --optimizer symplectic_muon --steps 2000 --batch-size 16 --wandb --wandb-project research-projekts --wandb-name symplectic-muon-t4


In [ ]:
# Step 6: Credit Protection & Automatic Machine Teardown
# @title Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}

if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning Colab GPU immediately to prevent idle credit burn...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Auto-kill skipped: not running inside interactive Colab: {e})")
